# LangChain: Memory

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

In [3]:
llm_model = "gpt-3.5-turbo-0301"

## ConversationBufferMemory
Esta memoria almacena todo el historial de la conversación en un buffer. Es útil cuando quieres tener acceso completo a todo lo que se ha dicho durante la interacción.
#### Características:
* Guarda todo el historial de la conversación.
* Ideal para casos donde necesitas contexto completo.
* Puede volverse grande si la conversación es larga.

## ConversationBufferWindowMemory
* Esta memoria es similar a la anterior, pero sólo guarda una ventana de las últimas n interacciones de la conversación. Es útil cuando necesitas contexto reciente sin sobrecargar la memoria.
#### Características:
* Almacena sólo las últimas n interacciones.
* Controlas el tamaño de la ventana (cuántas interacciones recordar).
* Más eficiente en términos de uso de memoria que ConversationBufferMemory.

## ConversationTokenBufferMemory
* Esta memoria funciona guardando el historial de la conversación, pero limitando el número de tokens almacenados. En lugar de contar las interacciones, cuenta los tokens (palabras o fragmentos de palabras) para mantener el contexto manejable.
#### Características:
* Guarda el historial basado en el número de tokens, no de interacciones.
* Mantiene el tamaño de la memoria limitado para no sobrepasar límites de tokens.
* Útil cuando se trabaja con modelos que tienen restricciones de tokens.

## ConversationSummaryMemory
En lugar de almacenar toda la conversación o una parte de ella, esta memoria guarda un resumen de la conversación. Se actualiza periódicamente para mantener un resumen conciso y relevante.
#### Características:
* Guarda un resumen en lugar de la conversación completa.
* Actualiza el resumen conforme avanza la conversación.
* Ideal para mantener el contexto general sin detalles excesivos.
* Menos carga de memoria, pero puede perder detalles específicos.

# ConversationBufferMemory

In [4]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

In [5]:
conversation.predict(input="Hola, mi nombre es Agustin")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hola, mi nombre es Agustin
AI:

> Finished chain.


'¡Hola Agustin! Soy un asistente de inteligencia artificial. ¿En qué puedo ayudarte hoy?'

In [6]:
conversation.predict(input="Cuantos goles hizo Messi?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hola, mi nombre es Agustin
AI: ¡Hola Agustin! Soy un asistente de inteligencia artificial. ¿En qué puedo ayudarte hoy?
Human: Cuantos goles hizo Messi?
AI:

> Finished chain.


'Messi ha marcado un total de 672 goles en su carrera profesional hasta el momento. ¿Te gustaría saber cuántos goles ha marcado en una temporada específica o en un torneo en particular?'

In [7]:
conversation.predict(input="Como me llamo?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hola, mi nombre es Agustin
AI: ¡Hola Agustin! Soy un asistente de inteligencia artificial. ¿En qué puedo ayudarte hoy?
Human: Cuantos goles hizo Messi?
AI: Messi ha marcado un total de 672 goles en su carrera profesional hasta el momento. ¿Te gustaría saber cuántos goles ha marcado en una temporada específica o en un torneo en particular?
Human: Como me llamo?
AI:

> Finished chain.


'Tu nombre es Agustin, como me lo dijiste al principio de nuestra conversación. ¿Hay algo más en lo que pueda ayudarte?'

In [8]:
print(memory.buffer)

Human: Hola, mi nombre es Agustin
AI: ¡Hola Agustin! Soy un asistente de inteligencia artificial. ¿En qué puedo ayudarte hoy?
Human: Cuantos goles hizo Messi?
AI: Messi ha marcado un total de 672 goles en su carrera profesional hasta el momento. ¿Te gustaría saber cuántos goles ha marcado en una temporada específica o en un torneo en particular?
Human: Como me llamo?
AI: Tu nombre es Agustin, como me lo dijiste al principio de nuestra conversación. ¿Hay algo más en lo que pueda ayudarte?


In [9]:
memory.load_memory_variables({})

{'history': 'Human: Hola, mi nombre es Agustin\nAI: ¡Hola Agustin! Soy un asistente de inteligencia artificial. ¿En qué puedo ayudarte hoy?\nHuman: Cuantos goles hizo Messi?\nAI: Messi ha marcado un total de 672 goles en su carrera profesional hasta el momento. ¿Te gustaría saber cuántos goles ha marcado en una temporada específica o en un torneo en particular?\nHuman: Como me llamo?\nAI: Tu nombre es Agustin, como me lo dijiste al principio de nuestra conversación. ¿Hay algo más en lo que pueda ayudarte?'}

In [10]:
memory = ConversationBufferMemory()

In [11]:
memory.save_context({"input": "Hola"}, 
                    {"output": "Hola, como estas?"})

In [12]:
print(memory.buffer)

Human: Hola
AI: Hola, como estas?


In [13]:
memory.load_memory_variables({})

{'history': 'Human: Hola\nAI: Hola, como estas?'}

In [14]:
memory.save_context({"input": "Muy bien, tomando un cafe"}, 
                    {"output": "Genial"})

In [15]:
memory.load_memory_variables({})

{'history': 'Human: Hola\nAI: Hola, como estas?\nHuman: Muy bien, tomando un cafe\nAI: Genial'}

# ConversationBufferWindowMemory

In [16]:
from langchain.memory import ConversationBufferWindowMemory

In [17]:
memory = ConversationBufferWindowMemory(k=1)               

In [18]:
memory.save_context({"input": "Hola"},
                    {"output": "Hola, Como Estas?"})
memory.save_context({"input": "Muy bien, tomando cafe"},
                    {"output": "Genial"})

In [19]:
memory.load_memory_variables({})

{'history': 'Human: Muy bien, tomando cafe\nAI: Genial'}

In [20]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=False
)

In [21]:
conversation.predict(input="Hola mi nombre es Agustin")

'¡Hola Agustin! Soy un asistente de inteligencia artificial. ¿En qué puedo ayudarte hoy?'

In [22]:
conversation.predict(input="Cuantos goles hizo Messi?")

'Messi ha marcado un total de 672 goles en su carrera profesional hasta el momento. ¿Te gustaría saber cuántos goles ha marcado en una temporada específica o en un torneo en particular?'

In [23]:
conversation.predict(input="Como me llamo?")

'Lo siento, no tengo acceso a esa información. ¿Hay algo más en lo que pueda ayudarte?'

# ConversationTokenBufferMemory

In [24]:
from langchain.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [25]:
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
memory.save_context({"input": "¿Qué es la IA?"},
                    {"output": "¡Increíble!"})
memory.save_context({"input": "¿Qué es la retropropagación?"},
                    {"output": "¡Hermosa!"})
memory.save_context({"input": "¿Qué son los chatbots?"}, 
                    {"output": "¡Encantadores!"})

In [26]:
memory.load_memory_variables({})

{'history': 'Human: ¿Qué es la retropropagación?\nAI: ¡Hermosa!\nHuman: ¿Qué son los chatbots?\nAI: ¡Encantadores!'}

# ConversationSummaryMemory

In [27]:
from langchain.memory import ConversationSummaryBufferMemory

In [28]:
horario = "Hay una reunión a las 8am con tu equipo de producto. \
Necesitarás tu presentación de PowerPoint preparada. \
De 9am a 12pm tienes tiempo para trabajar en tu proyecto de LangChain, \
lo cual irá rápido porque LangChain es una herramienta muy poderosa. \
Al mediodía, almuerzo en el restaurante italiano con un cliente que viene \
de más de una hora de distancia para reunirse contigo y entender lo último en IA. \
Asegúrate de llevar tu laptop para mostrar la última demostración de LLM."

memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory.save_context({"input": "Hola"}, {"output": "¿Qué tal?"})
memory.save_context({"input": "No mucho, solo pasando el rato"},
                    {"output": "Genial"})
memory.save_context({"input": "¿Qué hay en el horario de hoy?"}, 
                    {"output": f"{horario}"})


In [29]:
memory.load_memory_variables({})

{'history': "System: The human greets the AI in Spanish and asks what's on the schedule for the day. The AI informs the human of a morning meeting with their product team, followed by time to work on their LangChain project, which is a powerful tool. At noon, the human has a lunch meeting with a client interested in the latest in AI and should bring their laptop to show a demo of LLM."}

In [30]:
{
    "history": "Sistema: El humano saluda a la IA en español y pregunta qué hay en el horario del día. La IA informa al humano sobre una reunión matutina con su equipo de producto, seguida de tiempo para trabajar en su proyecto de LangChain, que es una herramienta poderosa. Al mediodía, el humano tiene una reunión para almorzar con un cliente interesado en lo último en IA y debe llevar su laptop para mostrar una demostración de LLM."
}

{'history': 'Sistema: El humano saluda a la IA en español y pregunta qué hay en el horario del día. La IA informa al humano sobre una reunión matutina con su equipo de producto, seguida de tiempo para trabajar en su proyecto de LangChain, que es una herramienta poderosa. Al mediodía, el humano tiene una reunión para almorzar con un cliente interesado en lo último en IA y debe llevar su laptop para mostrar una demostración de LLM.'}

In [31]:
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

In [32]:
conversation.predict(input="¿Cuál sería una buena demostración para mostrar?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human greets the AI in Spanish and asks what's on the schedule for the day. The AI informs the human of a morning meeting with their product team, followed by time to work on their LangChain project, which is a powerful tool. At noon, the human has a lunch meeting with a client interested in the latest in AI and should bring their laptop to show a demo of LLM.
Human: ¿Cuál sería una buena demostración para mostrar?
AI:

> Finished chain.


'Una buena demostración para mostrar sería la capacidad de LLM para analizar grandes cantidades de datos y proporcionar resultados precisos y útiles en tiempo real. También podría mostrar cómo LLM puede ser personalizado para satisfacer las necesidades específicas de un cliente y cómo puede integrarse fácilmente en su infraestructura existente.'

In [33]:
memory.load_memory_variables({})

{'history': "System: The human greets the AI in Spanish and asks what's on the schedule for the day. The AI informs the human of a morning meeting with their product team, followed by time to work on their LangChain project, which is a powerful tool. At noon, the human has a lunch meeting with a client interested in the latest in AI and should bring their laptop to show a demo of LLM.\nHuman: ¿Cuál sería una buena demostración para mostrar?\nAI: Una buena demostración para mostrar sería la capacidad de LLM para analizar grandes cantidades de datos y proporcionar resultados precisos y útiles en tiempo real. También podría mostrar cómo LLM puede ser personalizado para satisfacer las necesidades específicas de un cliente y cómo puede integrarse fácilmente en su infraestructura existente."}

In [34]:
{
    "history": "Sistema: El humano saluda a la IA en español y pregunta qué hay en el horario del día. La IA informa al humano sobre una reunión matutina con su equipo de producto, seguida de tiempo para trabajar en su proyecto de LangChain, que es una herramienta poderosa. Al mediodía, el humano tiene una reunión para almorzar con un cliente interesado en lo último en IA y debe llevar su laptop para mostrar una demostración de LLM.\nHumano: ¿Cuál sería una buena demostración para mostrar?\nIA: Una buena demostración para mostrar sería la capacidad de LLM para analizar grandes cantidades de datos y proporcionar resultados precisos y útiles en tiempo real. También podría mostrar cómo LLM puede ser personalizado para satisfacer las necesidades específicas de un cliente y cómo puede integrarse fácilmente en su infraestructura existente."
}

{'history': 'Sistema: El humano saluda a la IA en español y pregunta qué hay en el horario del día. La IA informa al humano sobre una reunión matutina con su equipo de producto, seguida de tiempo para trabajar en su proyecto de LangChain, que es una herramienta poderosa. Al mediodía, el humano tiene una reunión para almorzar con un cliente interesado en lo último en IA y debe llevar su laptop para mostrar una demostración de LLM.\nHumano: ¿Cuál sería una buena demostración para mostrar?\nIA: Una buena demostración para mostrar sería la capacidad de LLM para analizar grandes cantidades de datos y proporcionar resultados precisos y útiles en tiempo real. También podría mostrar cómo LLM puede ser personalizado para satisfacer las necesidades específicas de un cliente y cómo puede integrarse fácilmente en su infraestructura existente.'}